# FaceFinalML2 Colab Runner

Run the single code cell below in Google Colab with a GPU runtime. It clones the GitHub repo, downloads the Google Drive BMI zip, runs the leakage-free Face-to-BMI pipeline, and saves metrics under `outputs/metrics/`.

In [ ]:
# FaceFinalML2 Colab GPU runner: Face-to-BMI replication + modern embedding ensemble
# 1) In Colab: Runtime -> Change runtime type -> GPU, preferably A100.
# 2) Open this notebook from GitHub or upload it to Colab.
# 3) Run this single cell.
#
# This cell is self-contained. It clones/pulls the repo, downloads BMI.zip from
# Google Drive, audits the data, creates leakage-free slug/person splits, detects
# face crops, extracts frozen FaceNet/VGGFace2 + ConvNeXt + optional DINOv2
# embeddings, trains regularized regressors, evaluates against the paper's
# Pearson-r target, and writes outputs/metrics plus a Streamlit demo stub.
#
# It does not commit data or model artifacts to GitHub.

import os
import shlex
import subprocess
from pathlib import Path


def run(cmd, env=None, cwd=None):
    print(f"\n$ {cmd}", flush=True)
    p = subprocess.run(cmd, shell=True, env=env, cwd=cwd)
    if p.returncode != 0:
        raise SystemExit(f"Command failed with exit code {p.returncode}: {cmd}")


print("Checking GPU...")
subprocess.run("nvidia-smi", shell=True)

REPO = "manuelarceaguirre/facefinalml2"
DATA_FILE_ID = "16XA-MCnTG8ONdgxK0uPfFXWnA5oF3bFa"
WORKDIR = "/content/facefinalml2"
ZIP_PATH = f"{WORKDIR}/data/raw/BMI.zip"

if not os.path.isdir(WORKDIR):
    run(f"git clone https://github.com/{REPO}.git {shlex.quote(WORKDIR)}")
else:
    print(f"Repo already exists at {WORKDIR}; pulling latest...")
    run("git pull --ff-only || true", cwd=WORKDIR)

os.chdir(WORKDIR)

# Install a practical, Colab-friendly stack. InsightFace/ArcFace can be added later,
# but this runner avoids fragile native installs and uses facenet-pytorch + timm.
run("python -m pip install -q -U pip setuptools wheel")
run(
    "python -m pip install -q "
    "numpy pandas scipy scikit-learn xgboost joblib tqdm pillow matplotlib seaborn "
    "opencv-python-headless facenet-pytorch timm transformers accelerate "
    "gdown fastapi uvicorn python-multipart streamlit"
)

print("Authenticating to Google Drive for BMI.zip...")
from google.colab import auth

auth.authenticate_user()

from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

Path("data/raw").mkdir(parents=True, exist_ok=True)
Path("data/extracted").mkdir(parents=True, exist_ok=True)

if not os.path.exists(ZIP_PATH):
    service = build("drive", "v3")
    request = service.files().get_media(fileId=DATA_FILE_ID)
    with open(ZIP_PATH, "wb") as f:
        downloader = MediaIoBaseDownload(f, request)
        done = False
        while not done:
            status, done = downloader.next_chunk()
            if status:
                print(f"Drive download: {int(status.progress() * 100)}%", flush=True)
else:
    print("BMI.zip already exists; skipping download.")

run("unzip -q -o data/raw/BMI.zip -d data/extracted")
run("find data/extracted -maxdepth 4 -type f | head -50")

os.environ.update({
    "PYTHONUNBUFFERED": "1",
    "CUDA_LAUNCH_BLOCKING": "0",
})

runner_code = r'''#!/usr/bin/env python3
"""FaceFinalML2 Colab pipeline.

This script is written by notebooks/facefinalml2_colab_runner.ipynb inside Colab.
It performs the complete first-pass project pipeline:

1. Audit BMI data and metadata.
2. Build image-label rows.
3. Create leakage-free group splits.
4. Detect faces and write tight/loose crops.
5. Extract frozen embeddings from FaceNet/VGGFace2, ConvNeXt, and optional DINOv2.
6. Fit regularized regressors and a validation-weighted ensemble.
7. Save metrics, predictions, plots, and a Streamlit demo stub.
"""
from __future__ import annotations

import json
import math
import os
import random
import re
import time
import warnings
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple

import joblib
import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageOps
from scipy.stats import pearsonr, spearmanr
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import ElasticNetCV, RidgeCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=UserWarning)

ROOT = Path.cwd()
DATA_ROOT = ROOT / "data"
EXTRACTED = DATA_ROOT / "extracted"
OUT = ROOT / "outputs"
CROPS = OUT / "crops"
FEATURES = OUT / "features"
METRICS = OUT / "metrics"
MODELS = ROOT / "models"
FIGURES = OUT / "figures"

for p in [OUT, CROPS, FEATURES, METRICS, MODELS, FIGURES]:
    p.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"DEVICE={DEVICE}", flush=True)

PAPER_BASELINE = {
    "paper_vgg_net_svr_overall_r": 0.47,
    "paper_vgg_face_svr_overall_r": 0.65,
    "paper_vgg_face_male_r": 0.71,
    "paper_vgg_face_female_r": 0.57,
}


def norm_col(c: str) -> str:
    return (
        str(c).strip().lower()
        .replace(" ", "_")
        .replace("-", "_")
        .replace("/", "_")
        .replace("(", "")
        .replace(")", "")
        .replace("__", "_")
    )


def pick_col(df: pd.DataFrame, candidates: Iterable[str], required: bool = False) -> Optional[str]:
    columns = set(df.columns)
    for c in candidates:
        if c in columns:
            return c
    if required:
        raise ValueError(f"Missing required column among {list(candidates)}; available={df.columns.tolist()}")
    return None


def find_metadata_csv() -> Path:
    csvs = sorted(EXTRACTED.rglob("*.csv"), key=lambda p: p.stat().st_size, reverse=True)
    if not csvs:
        raise FileNotFoundError("No CSV found under data/extracted")
    scored = []
    for p in csvs:
        try:
            head = pd.read_csv(p, nrows=5)
            cols = [norm_col(c) for c in head.columns]
            score = 0
            joined = " ".join(cols + [p.name.lower()])
            for token in ["bmi", "weight", "height", "slug", "image", "photo"]:
                score += int(token in joined)
            scored.append((score, p.stat().st_size, p))
        except Exception:
            pass
    if not scored:
        return csvs[0]
    scored.sort(reverse=True)
    return scored[0][2]


def audit_and_build_rows() -> pd.DataFrame:
    meta_path = find_metadata_csv()
    print(f"Metadata CSV: {meta_path}", flush=True)
    df = pd.read_csv(meta_path)
    df = df.rename(columns={c: norm_col(c) for c in df.columns})
    print(f"Raw metadata shape={df.shape}")
    print(f"Columns={df.columns.tolist()}")
    print(df.head().to_string())

    slug_col = pick_col(df, ["slug", "id", "person_id", "subject", "subject_id", "name"], required=True)
    image_col = pick_col(df, ["image_path", "path", "filename", "file", "image", "photo", "img"])
    weight_col = pick_col(df, ["weight_kg", "weight", "current_weight", "weightkg"])
    height_col = pick_col(df, ["height_m", "height", "height_meters", "heightm"])
    bmi_col = pick_col(df, ["actual_bmi", "bmi", "body_mass_index"])
    gender_col = pick_col(df, ["gender", "sex"])
    type_col = pick_col(df, ["type", "category", "class"])

    image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    all_images = [p for p in EXTRACTED.rglob("*") if p.suffix.lower() in image_exts]
    print(f"Found {len(all_images)} image files", flush=True)
    if not all_images:
        raise FileNotFoundError("No image files found after unzip")

    by_parent = {}
    by_name = {}
    by_stem = {}
    for p in all_images:
        by_parent.setdefault(p.parent.name, []).append(p)
        by_name[p.name] = p
        by_stem[p.stem] = p

    def paths_for_row(row: pd.Series) -> List[Path]:
        if image_col is not None and pd.notna(row.get(image_col)):
            raw = str(row[image_col])
            name = Path(raw).name
            stem = Path(raw).stem
            if name in by_name:
                return [by_name[name]]
            if stem in by_stem:
                return [by_stem[stem]]
            hits = [p for p in all_images if name and name in str(p)]
            if hits:
                return sorted(hits)
        slug = str(row[slug_col])
        if slug in by_parent:
            return sorted(by_parent[slug])
        pat = re.escape(slug)
        hits = [p for p in all_images if re.search(pat, p.name) or re.search(pat, str(p.parent))]
        return sorted(hits)

    records = []
    for i, row in df.iterrows():
        paths = paths_for_row(row)
        for j, p in enumerate(paths):
            rec = row.to_dict()
            rec["source_row"] = int(i)
            rec["image_index_within_row"] = int(j)
            rec["image_path"] = str(p)
            rec["group_id"] = str(row[slug_col])
            records.append(rec)
    img_df = pd.DataFrame(records)
    print(f"Image-label rows before filtering={img_df.shape}", flush=True)
    if img_df.empty:
        raise RuntimeError("Could not map metadata rows to image files. Inspect the extracted data layout.")

    if bmi_col is not None:
        img_df["bmi"] = pd.to_numeric(img_df[bmi_col], errors="coerce")
    elif weight_col is not None and height_col is not None:
        weight = pd.to_numeric(img_df[weight_col], errors="coerce")
        height = pd.to_numeric(img_df[height_col], errors="coerce")
        height = np.where(height > 3.0, height / 100.0, height)
        img_df["bmi"] = weight / (height ** 2)
    else:
        raise ValueError("Need either an Actual BMI/BMI column or weight+height columns")

    img_df["bmi"] = pd.to_numeric(img_df["bmi"], errors="coerce")
    img_df = img_df[img_df["bmi"].between(13, 70)].copy()

    if height_col is not None:
        h = pd.to_numeric(img_df[height_col], errors="coerce")
        h = np.where(h > 3.0, h / 100.0, h)
        img_df["height_m_clean"] = h
    if weight_col is not None:
        img_df["weight_kg_clean"] = pd.to_numeric(img_df[weight_col], errors="coerce")

    if gender_col is not None:
        img_df["gender_clean"] = img_df[gender_col].astype(str).str.lower().str.strip()
    else:
        img_df["gender_clean"] = "unknown"

    if type_col is not None:
        img_df["type_clean"] = img_df[type_col].astype(str).str.lower().str.strip()

    def bmi_category(b: float) -> str:
        if b < 18.5:
            return "underweight"
        if b < 25:
            return "healthy"
        if b < 30:
            return "overweight"
        if b < 35:
            return "obesity_1"
        if b < 40:
            return "obesity_2"
        return "obesity_3"

    img_df["bmi_cat"] = img_df["bmi"].map(bmi_category)

    ok, widths, heights = [], [], []
    for p in tqdm(img_df["image_path"].tolist(), desc="verifying images"):
        try:
            with Image.open(p) as im:
                im.verify()
            with Image.open(p) as im:
                widths.append(im.size[0])
                heights.append(im.size[1])
            ok.append(True)
        except Exception:
            widths.append(np.nan)
            heights.append(np.nan)
            ok.append(False)
    img_df["image_ok"] = ok
    img_df["width"] = widths
    img_df["height_px"] = heights
    img_df = img_df[img_df["image_ok"]].copy().reset_index(drop=True)

    print("Clean rows:", img_df.shape)
    print(img_df[["image_path", "group_id", "bmi", "bmi_cat", "gender_clean"]].head().to_string())
    print("BMI summary:")
    print(img_df["bmi"].describe().to_string())
    print("Category counts:")
    print(img_df["bmi_cat"].value_counts().to_string())

    img_df.to_csv(OUT / "audit_image_rows.csv", index=False)
    return img_df


def make_splits(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy().reset_index(drop=True)
    df["strata"] = df["gender_clean"].astype(str) + "_" + df["bmi_cat"].astype(str)
    counts = df["strata"].value_counts()
    df.loc[df["strata"].map(counts) < 5, "strata"] = "rare"

    n_groups = df["group_id"].nunique()
    if n_groups < 10:
        raise RuntimeError(f"Too few groups for reliable leakage-free split: {n_groups}")

    try:
        sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
        trainval_idx, test_idx = next(sgkf.split(df, df["strata"], df["group_id"]))
        trainval = df.iloc[trainval_idx].copy()
        test = df.iloc[test_idx].copy()
        sgkf2 = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED + 1)
        train_idx_rel, val_idx_rel = next(sgkf2.split(trainval, trainval["strata"], trainval["group_id"]))
        train = trainval.iloc[train_idx_rel].copy()
        val = trainval.iloc[val_idx_rel].copy()
    except Exception as e:
        print(f"StratifiedGroupKFold failed ({e}); falling back to GroupShuffleSplit", flush=True)
        gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
        trainval_idx, test_idx = next(gss.split(df, groups=df["group_id"]))
        trainval = df.iloc[trainval_idx].copy()
        test = df.iloc[test_idx].copy()
        gss2 = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED + 1)
        train_idx_rel, val_idx_rel = next(gss2.split(trainval, groups=trainval["group_id"]))
        train = trainval.iloc[train_idx_rel].copy()
        val = trainval.iloc[val_idx_rel].copy()

    train["split"] = "train"
    val["split"] = "val"
    test["split"] = "test"
    splits = pd.concat([train, val, test], ignore_index=True)

    groups = {s: set(splits.loc[splits.split == s, "group_id"]) for s in ["train", "val", "test"]}
    assert groups["train"].isdisjoint(groups["val"])
    assert groups["train"].isdisjoint(groups["test"])
    assert groups["val"].isdisjoint(groups["test"])

    split_path = OUT / "split_seed42.csv"
    splits.to_csv(split_path, index=False)
    audit = splits.groupby("split").agg(
        n_images=("image_path", "count"),
        n_groups=("group_id", "nunique"),
        bmi_mean=("bmi", "mean"),
        bmi_std=("bmi", "std"),
        bmi_min=("bmi", "min"),
        bmi_max=("bmi", "max"),
    )
    print("Split audit:")
    print(audit.to_string())
    audit.to_csv(METRICS / "split_audit.csv")
    print("Category distribution by split:")
    print(pd.crosstab(splits["split"], splits["bmi_cat"], normalize="index").round(3).to_string())
    return splits


def square_expand_box(box, w: int, h: int, margin: float) -> Tuple[int, int, int, int]:
    x1, y1, x2, y2 = [float(v) for v in box]
    cx, cy = (x1 + x2) / 2.0, (y1 + y2) / 2.0
    side = max(x2 - x1, y2 - y1) * margin
    nx1 = int(max(0, round(cx - side / 2)))
    ny1 = int(max(0, round(cy - side / 2)))
    nx2 = int(min(w, round(cx + side / 2)))
    ny2 = int(min(h, round(cy + side / 2)))
    if nx2 <= nx1 or ny2 <= ny1:
        return 0, 0, w, h
    return nx1, ny1, nx2, ny2


def center_square_box(w: int, h: int) -> Tuple[int, int, int, int]:
    side = min(w, h)
    x1 = (w - side) // 2
    y1 = (h - side) // 2
    return x1, y1, x1 + side, y1 + side


def make_crops(splits: pd.DataFrame) -> pd.DataFrame:
    from facenet_pytorch import MTCNN

    crop_csv = OUT / "split_seed42_with_crops.csv"
    if crop_csv.exists():
        print(f"Using existing crops CSV: {crop_csv}")
        return pd.read_csv(crop_csv)

    tight_dir = CROPS / "tight_160"
    loose_dir = CROPS / "loose_224"
    tight_dir.mkdir(parents=True, exist_ok=True)
    loose_dir.mkdir(parents=True, exist_ok=True)

    mtcnn = MTCNN(keep_all=True, device=DEVICE)
    rows = []
    for idx, row in tqdm(splits.iterrows(), total=len(splits), desc="detecting/cropping faces"):
        path = Path(row["image_path"])
        image = Image.open(path).convert("RGB")
        image = ImageOps.exif_transpose(image)
        w, h = image.size
        face_detected = False
        det_score = 0.0
        chosen_box = center_square_box(w, h)
        try:
            boxes, probs = mtcnn.detect(image)
            if boxes is not None and len(boxes) > 0:
                scores = []
                for b, pr in zip(boxes, probs):
                    x1, y1, x2, y2 = b
                    area = max(1.0, (x2 - x1) * (y2 - y1)) / max(1.0, w * h)
                    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
                    dist = math.sqrt(((cx - w / 2) / w) ** 2 + ((cy - h / 2) / h) ** 2)
                    scores.append(float(pr or 0) + 0.25 * area - 0.10 * dist)
                k = int(np.argmax(scores))
                chosen_box = boxes[k]
                det_score = float(probs[k] or 0)
                face_detected = True
        except Exception as e:
            print(f"MTCNN failed for {path}: {e}")

        tight_box = square_expand_box(chosen_box, w, h, margin=1.15) if face_detected else center_square_box(w, h)
        loose_box = square_expand_box(chosen_box, w, h, margin=1.55) if face_detected else center_square_box(w, h)

        stem = f"{idx:06d}_{re.sub(r'[^A-Za-z0-9_.-]+', '_', path.stem)[:80]}"
        tight_path = tight_dir / f"{stem}.jpg"
        loose_path = loose_dir / f"{stem}.jpg"

        image.crop(tight_box).resize((160, 160), Image.BICUBIC).save(tight_path, quality=95)
        image.crop(loose_box).resize((224, 224), Image.BICUBIC).save(loose_path, quality=95)

        rec = row.to_dict()
        rec.update({
            "tight_crop_path": str(tight_path),
            "loose_crop_path": str(loose_path),
            "face_detected": bool(face_detected),
            "det_score": float(det_score),
            "tight_box": json.dumps([int(x) for x in tight_box]),
            "loose_box": json.dumps([int(x) for x in loose_box]),
        })
        rows.append(rec)

    out = pd.DataFrame(rows)
    out.to_csv(crop_csv, index=False)
    print(f"Face detection rate: {out['face_detected'].mean():.3f}")
    return out


class PathDataset(Dataset):
    def __init__(self, paths: List[str], transform):
        self.paths = list(paths)
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        return self.transform(img)


def extract_features(name: str, model: torch.nn.Module, paths: List[str], transform, batch_size: int = 96) -> np.ndarray:
    cache = FEATURES / f"{name}.npy"
    if cache.exists():
        print(f"Loading cached features {cache}")
        return np.load(cache)
    ds = PathDataset(paths, transform)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=(DEVICE == "cuda"))
    feats = []
    model = model.to(DEVICE).eval()
    with torch.no_grad():
        for x in tqdm(loader, desc=f"extracting {name}"):
            x = x.to(DEVICE, non_blocking=True)
            y = model(x)
            if isinstance(y, (tuple, list)):
                y = y[0]
            feats.append(y.detach().float().cpu().numpy())
    X = np.concatenate(feats, axis=0)
    np.save(cache, X)
    print(f"{name}: {X.shape}")
    return X


def build_feature_sets(df: pd.DataFrame) -> Dict[str, np.ndarray]:
    import timm
    from facenet_pytorch import InceptionResnetV1, fixed_image_standardization
    from timm.data import create_transform, resolve_data_config

    feature_sets = {}

    facenet = InceptionResnetV1(pretrained="vggface2").eval()
    facenet_transform = transforms.Compose([
        transforms.Resize((160, 160)),
        transforms.ToTensor(),
        fixed_image_standardization,
    ])
    feature_sets["facenet_vggface2_tight"] = extract_features(
        "facenet_vggface2_tight",
        facenet,
        df["tight_crop_path"].tolist(),
        facenet_transform,
        batch_size=128,
    )

    conv_name_candidates = ["convnext_tiny.fb_in22k_ft_in1k", "convnext_tiny"]
    conv_model = None
    conv_name = None
    for candidate in conv_name_candidates:
        try:
            conv_model = timm.create_model(candidate, pretrained=True, num_classes=0, global_pool="avg")
            conv_name = candidate
            break
        except Exception as e:
            print(f"Could not load {candidate}: {e}")
    if conv_model is not None:
        cfg = resolve_data_config({}, model=conv_model)
        conv_transform = create_transform(**cfg)
        feature_sets["convnext_loose"] = extract_features(
            "convnext_loose",
            conv_model,
            df["loose_crop_path"].tolist(),
            conv_transform,
            batch_size=96,
        )
        print(f"Loaded ConvNeXt model: {conv_name}")

    try:
        dino = torch.hub.load("facebookresearch/dinov2", "dinov2_vits14", pretrained=True)
        dino_transform = transforms.Compose([
            transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.BICUBIC),
            transforms.ToTensor(),
            transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ])
        feature_sets["dinov2_vits14_loose"] = extract_features(
            "dinov2_vits14_loose",
            dino,
            df["loose_crop_path"].tolist(),
            dino_transform,
            batch_size=96,
        )
    except Exception as e:
        print(f"DINOv2 extraction skipped because loading failed: {e}", flush=True)

    if len(feature_sets) >= 2:
        ordered = [feature_sets[k] for k in sorted(feature_sets.keys())]
        feature_sets["concat_all"] = np.concatenate(ordered, axis=1)
        np.save(FEATURES / "concat_all.npy", feature_sets["concat_all"])
        print(f"concat_all: {feature_sets['concat_all'].shape}")

    return feature_sets


def safe_corr(fn, y_true, y_pred) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    if len(y_true) < 3 or np.std(y_true) == 0 or np.std(y_pred) == 0:
        return float("nan")
    try:
        return float(fn(y_true, y_pred)[0] if fn is pearsonr else fn(y_true, y_pred).correlation)
    except Exception:
        return float("nan")


def regression_metrics(y_true, y_pred) -> Dict[str, float]:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    return {
        "pearson_r": safe_corr(pearsonr, y_true, y_pred),
        "spearman_rho": safe_corr(spearmanr, y_true, y_pred),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": rmse,
        "r2": float(r2_score(y_true, y_pred)),
        "bias": float(np.mean(y_pred - y_true)),
        "within_2": float(np.mean(np.abs(y_pred - y_true) <= 2.0)),
        "within_5": float(np.mean(np.abs(y_pred - y_true) <= 5.0)),
    }


def make_regressors(n_features: int) -> Dict[str, object]:
    regs = {
        "ridge": make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-4, 4, 41))),
        "elastic": make_pipeline(
            StandardScaler(),
            ElasticNetCV(
                alphas=np.logspace(-4, 2, 25),
                l1_ratio=[0.05, 0.1, 0.3, 0.5, 0.8],
                max_iter=20000,
                cv=5,
                random_state=SEED,
            ),
        ),
        "pca_svr": make_pipeline(
            StandardScaler(),
            PCA(n_components=min(256, max(2, n_features - 1)), random_state=SEED),
            SVR(kernel="rbf", C=10.0, epsilon=0.3, gamma="scale"),
        ),
        "rf": make_pipeline(
            StandardScaler(),
            RandomForestRegressor(n_estimators=400, max_depth=8, min_samples_leaf=3, random_state=SEED, n_jobs=-1),
        ),
    }
    try:
        from xgboost import XGBRegressor
        regs["xgb"] = make_pipeline(
            StandardScaler(),
            XGBRegressor(
                n_estimators=700,
                max_depth=3,
                learning_rate=0.03,
                subsample=0.85,
                colsample_bytree=0.85,
                objective="reg:squarederror",
                random_state=SEED,
                n_jobs=-1,
            ),
        )
    except Exception as e:
        print(f"XGBoost skipped: {e}")
    return regs


def train_regressors(df: pd.DataFrame, feature_sets: Dict[str, np.ndarray]) -> Tuple[pd.DataFrame, List[dict]]:
    y = df["bmi"].to_numpy(float)
    train_mask = df["split"].eq("train").to_numpy()
    val_mask = df["split"].eq("val").to_numpy()
    test_mask = df["split"].eq("test").to_numpy()

    records = []
    fitted = []
    for feat_name, X in feature_sets.items():
        X_train, X_val, X_test = X[train_mask], X[val_mask], X[test_mask]
        y_train, y_val, y_test = y[train_mask], y[val_mask], y[test_mask]
        regs = make_regressors(X_train.shape[1])
        for reg_name, reg in regs.items():
            tag = f"{feat_name}__{reg_name}"
            print(f"\nTraining {tag} X={X_train.shape}", flush=True)
            try:
                reg.fit(X_train, y_train)
                pred_val = reg.predict(X_val)
                pred_test = reg.predict(X_test)
                val_m = regression_metrics(y_val, pred_val)
                test_m = regression_metrics(y_test, pred_test)
                records.append({
                    "model": tag,
                    "feature_set": feat_name,
                    "regressor": reg_name,
                    "split": "val",
                    **val_m,
                })
                records.append({
                    "model": tag,
                    "feature_set": feat_name,
                    "regressor": reg_name,
                    "split": "test",
                    **test_m,
                })
                joblib.dump(reg, MODELS / f"{tag}.joblib")
                fitted.append({
                    "tag": tag,
                    "feature_set": feat_name,
                    "regressor": reg_name,
                    "model": reg,
                    "val_pred": pred_val,
                    "test_pred": pred_test,
                    "val_pearson": val_m["pearson_r"],
                })
                print(f"{tag}: val r={val_m['pearson_r']:.4f} test r={test_m['pearson_r']:.4f} MAE={test_m['mae']:.3f}", flush=True)
            except Exception as e:
                print(f"FAILED {tag}: {e}", flush=True)

    results = pd.DataFrame(records)
    results.to_csv(METRICS / "model_results_long.csv", index=False)
    if not results.empty:
        wide = results.pivot_table(index="model", columns="split", values=["pearson_r", "mae", "rmse", "r2"], aggfunc="first")
        wide.to_csv(METRICS / "model_results_wide.csv")
        print("\nTop validation models:")
        print(results[results.split == "val"].sort_values("pearson_r", ascending=False).head(15).to_string(index=False))
    return results, fitted


def ensemble_and_subgroups(df: pd.DataFrame, fitted: List[dict]) -> Dict[str, object]:
    val_df = df[df.split == "val"].reset_index(drop=True)
    test_df = df[df.split == "test"].reset_index(drop=True)
    y_val = val_df["bmi"].to_numpy(float)
    y_test = test_df["bmi"].to_numpy(float)

    usable = [m for m in fitted if np.isfinite(m["val_pearson"]) and m["val_pearson"] > 0]
    usable = sorted(usable, key=lambda m: m["val_pearson"], reverse=True)[:8]
    if not usable:
        raise RuntimeError("No usable fitted models for ensemble")

    weights = np.array([max(0.0, m["val_pearson"]) ** 2 for m in usable], dtype=float)
    weights = weights / weights.sum()
    val_pred = sum(w * m["val_pred"] for w, m in zip(weights, usable))
    test_pred = sum(w * m["test_pred"] for w, m in zip(weights, usable))

    payload = {
        "paper_baseline": PAPER_BASELINE,
        "ensemble_members": [
            {"tag": m["tag"], "val_pearson": float(m["val_pearson"]), "weight": float(w)}
            for w, m in zip(weights, usable)
        ],
        "ensemble_val": regression_metrics(y_val, val_pred),
        "ensemble_test": regression_metrics(y_test, test_pred),
    }

    pred_rows = []
    for split_name, part_df, pred in [("val", val_df, val_pred), ("test", test_df, test_pred)]:
        tmp = part_df[["image_path", "group_id", "bmi", "bmi_cat", "gender_clean", "face_detected", "det_score"]].copy()
        tmp["split"] = split_name
        tmp["pred_bmi"] = pred
        tmp["abs_error"] = (tmp["pred_bmi"] - tmp["bmi"]).abs()
        pred_rows.append(tmp)
    pred_df = pd.concat(pred_rows, ignore_index=True)
    pred_df.to_csv(METRICS / "ensemble_predictions.csv", index=False)

    subgroup_records = []
    for split_name, part in pred_df.groupby("split"):
        for col in ["gender_clean", "bmi_cat", "face_detected"]:
            for value, g in part.groupby(col):
                if len(g) >= 5:
                    subgroup_records.append({
                        "split": split_name,
                        "subgroup_col": col,
                        "subgroup": str(value),
                        "n": int(len(g)),
                        **regression_metrics(g["bmi"], g["pred_bmi"]),
                    })
    subgroup_df = pd.DataFrame(subgroup_records)
    subgroup_df.to_csv(METRICS / "ensemble_subgroup_metrics.csv", index=False)

    with open(METRICS / "final_summary.json", "w") as f:
        json.dump(payload, f, indent=2, sort_keys=True)

    print("\nFINAL ENSEMBLE SUMMARY")
    print(json.dumps(payload, indent=2, sort_keys=True))
    if not subgroup_df.empty:
        print("\nSubgroup metrics:")
        print(subgroup_df.sort_values(["split", "subgroup_col", "subgroup"]).to_string(index=False))
    return payload


def write_streamlit_demo() -> None:
    demo = r'''
import streamlit as st
from PIL import Image

st.title("Face-to-BMI Educational Demo")
st.warning(
    "Academic demo only. This is not a medical diagnostic tool. "
    "BMI-from-face prediction is noisy, biased, and privacy-sensitive. "
    "Do not use it for health, employment, insurance, or personal judgments."
)

st.write(
    "This repository's Colab runner trains the final ensemble and writes metrics under `outputs/metrics`. "
    "For a production-quality demo, load the saved regressors and reuse the same crop/embedding functions "
    "from the Colab runner. This stub is intentionally conservative so the report can show the interface."
)

uploaded = st.file_uploader("Upload a face image", type=["jpg", "jpeg", "png"])
camera = st.camera_input("Or take a webcam photo")
source = uploaded or camera
if source is not None:
    image = Image.open(source).convert("RGB")
    st.image(image, caption="Input image", width=320)
    st.info("Connect this UI to the exported model bundle after final model selection.")
'''
    path = OUT / "streamlit_demo_stub.py"
    path.write_text(demo)
    print(f"Wrote demo stub: {path}")


def main() -> None:
    start = time.time()
    df = audit_and_build_rows()
    splits = make_splits(df)
    crop_df = make_crops(splits)
    feature_sets = build_feature_sets(crop_df)
    results, fitted = train_regressors(crop_df, feature_sets)
    summary = ensemble_and_subgroups(crop_df, fitted)
    write_streamlit_demo()
    elapsed = (time.time() - start) / 60.0
    print(f"\nDone in {elapsed:.1f} minutes")
    print("Important outputs:")
    print(f"  {OUT / 'split_seed42_with_crops.csv'}")
    print(f"  {METRICS / 'model_results_long.csv'}")
    print(f"  {METRICS / 'final_summary.json'}")
    print(f"  {METRICS / 'ensemble_predictions.csv'}")
    print(f"  {METRICS / 'ensemble_subgroup_metrics.csv'}")
    print(f"  {OUT / 'streamlit_demo_stub.py'}")
    print("Paper target: overall Pearson r > 0.65 on the held-out leakage-free test split.")


if __name__ == "__main__":
    main()
'''

Path("run_face_bmi_pipeline.py").write_text(runner_code)

print("\nStarting Face-to-BMI Colab pipeline.")
print("This writes:")
print("  face_bmi_pipeline.out")
print("  outputs/split_seed42_with_crops.csv")
print("  outputs/metrics/model_results_long.csv")
print("  outputs/metrics/final_summary.json")
print("  outputs/metrics/ensemble_predictions.csv")
print("  outputs/streamlit_demo_stub.py")
print("It does not commit raw data or model artifacts.\n")

env = os.environ.copy()
with open("face_bmi_pipeline.out", "a", buffering=1) as log:
    p = subprocess.Popen(
        ["bash", "-lc", "python3 run_face_bmi_pipeline.py"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        env=env,
    )
    assert p.stdout is not None
    for line in p.stdout:
        print(line, end="")
        log.write(line)
    rc = p.wait()
    if rc != 0:
        raise SystemExit(f"Face-to-BMI pipeline failed with exit code {rc}")

print("\nDone. Download or inspect these from the Colab file browser if needed:")
print("  face_bmi_pipeline.out")
print("  outputs/metrics/final_summary.json")
print("  outputs/metrics/model_results_long.csv")
print("  outputs/metrics/ensemble_predictions.csv")
print("  outputs/streamlit_demo_stub.py")

try:
    import json
    summary = json.loads(Path("outputs/metrics/final_summary.json").read_text())
    print("\nFINAL SUMMARY")
    print(json.dumps(summary, indent=2, sort_keys=True)[:12000])
except Exception as e:
    print(f"Could not print final JSON summary: {e}")
